In [8]:
import cv2
import mediapipe as mp
import numpy as np
import os

# ------------------------------
# Setup
# ------------------------------
DATA_DIR = "data/raw"
os.makedirs(DATA_DIR, exist_ok=True)

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
hands = mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.7, min_tracking_confidence=0.7)

import sys
print("Python:", sys.executable)
print("OpenCV:", cv2.__file__)
if "WIN32UI" not in cv2.getBuildInformation():
    raise RuntimeError("OpenCV has no Windows GUI support. Install only opencv-contrib-python in this environment, then restart the notebook kernel.")
print("GUI backend: WIN32UI")

print("👉 Press a letter key (A–Z) to save a sample. Press ESC to quit.")

# ------------------------------
# Helper: normalize landmarks
# ------------------------------
def normalize_landmarks(landmarks):
    wrist = landmarks[0]
    normalized = []
    for lm in landmarks:
        normalized.append([
            lm.x - wrist.x,
            lm.y - wrist.y,
            lm.z - wrist.z
        ])
    return np.array(normalized).flatten()

# ------------------------------
# Webcam loop
# ------------------------------
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
if not cap.isOpened():
    print("Camera could not be opened. Check Windows privacy settings or close other camera apps.")
else:
    print("Camera opened:", cap.getBackendName())

last_key_pressed = -1
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)  # mirror image
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(rgb)
    if result.multi_hand_landmarks:
        for hand_landmarks in result.multi_hand_landmarks:
            # Draw hand on screen
            mp_drawing.draw_landmarks(
                frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

# Show the frame and read keys from the OpenCV window
    try:
        cv2.imshow("Hand Sign Data Collection", frame)
        key_pressed = cv2.waitKey(1) & 0xFF
    except cv2.error as exc:
        print("OpenCV GUI error:", exc)
        break

    if key_pressed == 27:
        break
    if key_pressed == 255:
        last_key_pressed = -1
    elif (97 <= key_pressed <= 122 or 65 <= key_pressed <= 90) and key_pressed != last_key_pressed:
        last_key_pressed = key_pressed
        if result.multi_hand_landmarks:
            letter = chr(key_pressed).upper()
            landmarks = normalize_landmarks(result.multi_hand_landmarks[0].landmark)
            path = os.path.join(DATA_DIR, f"{letter}.npy")
            data = np.load(path).tolist() if os.path.exists(path) else []
            data.append(landmarks.tolist())
            np.save(path, np.array(data, dtype=np.float32))
            print(f"✅ Saved one sample for {letter} (total: {len(data)})")
        else:
            print("No hand detected; sample not saved. Press the letter again with your hand visible.")

cap.release()
try:
    cv2.destroyAllWindows()
except cv2.error:
    pass
hands.close()


Python: e:\Hand-sign-project-main\.venv\Scripts\python.exe
OpenCV: e:\Hand-sign-project-main\.venv\Lib\site-packages\cv2\__init__.py
GUI backend: WIN32UI
👉 Press a letter key (A–Z) to save a sample. Press ESC to quit.
Camera opened: DSHOW
✅ Saved one sample for E (total: 100)
✅ Saved one sample for E (total: 101)
✅ Saved one sample for E (total: 102)
✅ Saved one sample for E (total: 103)
✅ Saved one sample for E (total: 104)
✅ Saved one sample for E (total: 105)
✅ Saved one sample for E (total: 106)
✅ Saved one sample for E (total: 107)
✅ Saved one sample for E (total: 108)
✅ Saved one sample for E (total: 109)
✅ Saved one sample for E (total: 110)
✅ Saved one sample for E (total: 111)
✅ Saved one sample for E (total: 112)
✅ Saved one sample for E (total: 113)
✅ Saved one sample for E (total: 114)
✅ Saved one sample for E (total: 115)
✅ Saved one sample for E (total: 116)
✅ Saved one sample for E (total: 117)
✅ Saved one sample for E (total: 118)
✅ Saved one sample for E (total: 119)
✅

In [9]:
import sys, cv2
print(sys.executable)
print(cv2.__version__)
print("WIN32UI", "WIN32UI" in cv2.getBuildInformation())

e:\Hand-sign-project-main\.venv\Scripts\python.exe
4.11.0
WIN32UI True
